---
title: "EGC5310 — Semana 01: Estruturas de Dados como decisões de projeto"
subtitle: "Do problema aos dados, das operações à escolha da estrutura"
author: "Prof. Vinicius Ramos"
format:
  revealjs:
    slide-number: true
    chalkboard: true
    controls: true
    progress: true
    transition: fade
    code-overflow: wrap
    center: false
jupyter: python3
execute:
  echo: true
  warning: false
---

# EGC5310 — Semana 01

## Estruturas de Dados como decisões de projeto

**Ideia central da aula:** uma estrutura de dados não é apenas uma forma de armazenar valores.  
Ela determina quais operações serão naturais, custosas ou inadequadas.

> Cenário longitudinal da primeira metade do semestre: **Sistema Acadêmico**.

## Como usaremos este Notebook

Este arquivo é a **fonte única da aula**.

Ele reúne:

- narrativa e conceitos;
- exemplos executáveis;
- perguntas para discussão;
- pequenos experimentos;
- atividades para os estudantes;
- marcações que permitem gerar uma apresentação a partir do próprio Notebook.

Durante a aula, podemos alternar entre **modo apresentação** e **modo Notebook**, sem manter dois materiais independentes.

## Objetivos de aprendizagem

Ao final desta aula, espera-se que o estudante consiga:

1. reconhecer que a escolha de uma estrutura de dados depende das **operações exigidas pelo problema**;
2. distinguir, em nível introdutório, **dados**, **estrutura**, **operação** e **algoritmo**;
3. perceber que diferentes representações do mesmo conjunto de dados produzem custos computacionais distintos;
4. interpretar experimentalmente uma diferença de desempenho;
5. formular perguntas sobre custo computacional antes de escolher uma solução.

## O problema que inicia a disciplina

Considere um sistema acadêmico com milhares de estudantes.

Cada estudante possui, entre outros atributos:

- matrícula;
- nome;
- curso;
- fase;
- situação acadêmica.

A secretaria precisa executar operações diferentes sobre esses dados.

## Operações importam

Considere algumas necessidades do sistema:

- listar todos os estudantes;
- localizar um estudante pela matrícula;
- inserir um novo estudante;
- remover uma matrícula cancelada;
- percorrer estudantes em determinada ordem;
- localizar estudantes segundo algum critério;
- manter relações entre disciplinas e seus pré-requisitos.

**Pergunta:** existe uma única estrutura que seja igualmente boa para todas essas operações?

## Modelo inicial: registros de estudantes

Começaremos com uma representação deliberadamente simples.

Por enquanto, cada estudante será um pequeno registro Python. O objetivo não é discutir modelagem orientada a objetos, mas observar o efeito da **estrutura que organiza os registros**.

In [ ]:
estudantes = [
    {"matricula": 20260001, "nome": "Ana", "curso": "Ciência de Dados"},
    {"matricula": 20260002, "nome": "Bruno", "curso": "Ciência de Dados"},
    {"matricula": 20260003, "nome": "Carla", "curso": "Ciência de Dados"},
    {"matricula": 20260004, "nome": "Diego", "curso": "Ciência de Dados"},
]

estudantes

## Primeira tarefa

Precisamos localizar um estudante pela matrícula.

Com os dados organizados em uma lista, uma estratégia imediata é percorrer os registros até encontrar a matrícula desejada.

Antes de executar:

> Quantos estudantes poderemos precisar examinar?

In [ ]:
def buscar_por_matricula_lista(estudantes, matricula):
    for estudante in estudantes:
        if estudante["matricula"] == matricula:
            return estudante
    return None

buscar_por_matricula_lista(estudantes, 20260004)

## O que o algoritmo está fazendo?

Para uma busca em uma lista não ordenada:

- melhor caso: o elemento está no início;
- pior caso: o elemento está no final ou não existe;
- à medida que o número de registros cresce, o número potencial de comparações também cresce.

Ainda não precisamos formalizar toda a análise matemática.

A pergunta importante é:

> **como o custo da solução cresce quando o tamanho do problema cresce?**

## Uma representação alternativa

Se a operação dominante do sistema for **buscar diretamente pela matrícula**, podemos organizar os mesmos registros usando a matrícula como chave.

Observe que não mudamos o problema nem os estudantes.

Mudamos a **estrutura de acesso aos dados**.

In [ ]:
estudantes_por_matricula = {
    estudante["matricula"]: estudante
    for estudante in estudantes
}

estudantes_por_matricula

In [ ]:
def buscar_por_matricula_indice(indice, matricula):
    return indice.get(matricula)

buscar_por_matricula_indice(estudantes_por_matricula, 20260004)

## Mesmos dados, operações diferentes

Temos agora duas organizações possíveis:

**Lista**

- simples para percorrer sequencialmente;
- preserva uma sequência;
- busca por matrícula pode exigir várias comparações.

**Estrutura indexada por matrícula**

- exige uma organização adicional;
- favorece acesso direto pela chave;
- envolve outros custos e outras propriedades.

A disciplina estudará justamente esses compromissos.

## Um primeiro experimento

Vamos ampliar artificialmente o número de estudantes e comparar o tempo de busca.

O experimento não substitui a análise teórica. Ele serve para produzir uma **evidência observável** que depois precisaremos explicar.

In [ ]:
def gerar_estudantes(n):
    return [
        {
            "matricula": 20260000 + i,
            "nome": f"Estudante {i}",
            "curso": "Ciência de Dados"
        }
        for i in range(1, n + 1)
    ]

N = 100_000
base = gerar_estudantes(N)
indice = {e["matricula"]: e for e in base}

matricula_alvo = 20260000 + N

In [ ]:
import timeit

tempo_lista = timeit.timeit(
    lambda: buscar_por_matricula_lista(base, matricula_alvo),
    number=100
)

tempo_indice = timeit.timeit(
    lambda: buscar_por_matricula_indice(indice, matricula_alvo),
    number=100
)

print(f"Lista:  {tempo_lista:.6f} s")
print(f"Índice: {tempo_indice:.6f} s")
print(f"Razão lista/índice: {tempo_lista / tempo_indice:.1f}x")

## Cuidado ao interpretar benchmarks

Um benchmark responde:

> “o que aconteceu nesta execução, neste ambiente e com estes dados?”

A análise de complexidade busca responder algo mais geral:

> “como o custo tende a crescer à medida que o tamanho da entrada cresce?”

Ao longo da disciplina, usaremos **as duas perspectivas**:

1. raciocínio analítico;
2. experimentação computacional.

## Ordem de grandeza: primeira intuição

Sem aprofundar ainda a notação:

- uma operação que percorre aproximadamente todos os `n` elementos cresce de forma **linear**;
- uma operação cujo número de passos permanece aproximadamente independente de `n` tem comportamento **constante**, sob hipóteses apropriadas;
- outras estruturas produzirão outros padrões de crescimento.

Nas próximas semanas, vamos relacionar cada estrutura às operações que ela torna eficiente.

## O método da disciplina

Para cada nova estrutura, seguiremos aproximadamente o mesmo ciclo:

**Problema → operações → representação → algoritmo → custo → experimento → decisão**

Não começaremos pela definição abstrata da estrutura.

Começaremos por uma necessidade concreta de manipulação de dados e perguntaremos:

> **qual estrutura nos ajuda a resolver este problema e por quê?**

## Cenário longitudinal: Sistema Acadêmico

Na primeira metade do semestre, o **Sistema Acadêmico** será nosso cenário recorrente.

A cada semana, o sistema apresentará um novo problema de dados.

Não construiremos um sistema acadêmico completo.

Usaremos o cenário para estudar, de forma incremental:

- sequências;
- pilhas e filas;
- estruturas de associação;
- árvores;
- grafos;
- algoritmos de busca e ordenação;
- custos computacionais associados.

## Atividade em duplas ou trios

Para cada operação abaixo, discuta quais características uma estrutura de dados deveria oferecer:

1. encontrar rapidamente um estudante pela matrícula;
2. atender solicitações de matrícula na ordem de chegada;
3. permitir “desfazer” as últimas alterações realizadas;
4. representar pré-requisitos entre disciplinas;
5. manter estudantes ordenados por algum critério.

**Não é necessário saber ainda o nome da estrutura correta.**

O objetivo é descrever as **propriedades necessárias**.

## Registro da atividade

Preencha a tabela conceitualmente antes de pensar em código.

| Operação | O que precisa ser eficiente? | Que propriedade seria útil? |
|---|---|---|
| Buscar por matrícula | | |
| Processar solicitações em ordem de chegada | | |
| Desfazer alterações | | |
| Representar pré-requisitos | | |
| Manter dados ordenados | | |

A discussão será retomada ao longo das próximas semanas.

## Mini-desafio de código

Modifique a função de busca sequencial para também devolver o número de comparações realizadas.

Depois teste:

- matrícula do primeiro estudante;
- matrícula de um estudante no meio;
- matrícula do último estudante;
- matrícula inexistente.

**Pergunta:** como os resultados observados se relacionam com melhor caso, caso médio e pior caso?

In [ ]:
def buscar_com_contagem(estudantes, matricula):
    comparacoes = 0

    # TODO: complete a implementação.
    # A função deve retornar uma tupla:
    # (estudante_encontrado_ou_None, numero_de_comparacoes)

    pass

## Projeto integrador da disciplina

O projeto integrador não será um “grande sistema” separado das aulas.

Ele será construído a partir das decisões tomadas semanalmente.

A cada problema, registraremos:

- qual operação precisava ser resolvida;
- qual estrutura foi considerada;
- qual algoritmo foi utilizado;
- qual custo esperamos;
- que evidência experimental obtivemos;
- quais compromissos a solução introduziu.

Ao final, teremos uma trajetória de decisões de projeto baseada em estruturas de dados.

## Fechamento

A ideia fundamental desta primeira aula é simples:

> **não existe estrutura de dados “melhor” em termos absolutos; existe uma estrutura mais adequada para determinadas operações e restrições.**

Nas próximas aulas, transformaremos essa ideia em repertório técnico.

## Para a próxima etapa

No restante da Semana 01, vamos consolidar:

- ambiente de execução;
- experimentos com crescimento de entrada;
- leitura inicial de complexidade;
- exercícios curtos;
- registro das primeiras decisões do cenário do Sistema Acadêmico.

Este mesmo Notebook será evoluído como **material didático executável** e como **fonte da apresentação**.